<a href="https://colab.research.google.com/github/chamudithamk/Data-Driven-Dynamics-and-Control/blob/main/Dynamical_Systems_with_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Dynamical Systems with Machine Learning**

### **Overview of Dynamical Systems**

Traditionally, the dynamics of a system has been derived using Newtonian, Lagrangian or Hamiltonian first principles.

General form of a dynamical equation:

$$\dot{x} = f(x,t,u; \beta) + d$$

where $d$ is disturbance and $\beta$ is a system paramter that changes dynamic behaviour of a system (e.g., altitude of a aeroplane).

***NOTE**: $\beta$ can be used as a design parameter to control a system whereas $u$ is typically used in real-time feedback controllers*

However, first princples are only models of reality, sometimes they are extremely good and sometimes, $f$ is simply unknown. This is where data-driven approaches come into play. Data-driven approaches can tackle a number of other complexities as well: nonlinearity, uncertainty, high-dimensionality, stochasticity, chaos (highly sensitive to initial conditions and other parameters), transients, multi-scale (e.g., turbulence scales).

The measurements of such a dynamical system:

$$y = g(x,t) + n$$

where $n$ is noise. $y$ is not necessarily full-state and is estimated through an observer (e.g., Kalman Filter).

Ultimately, they can be used to predict future states, control and for design and optimisation purposes. Also, it is important that ML models are generalisable and interpretable rather than black-box models.

#### Example: Simulation of Lorenz Equations

It is a set of three coupled nonlinear ordinary differential equations founded by Edward Lorenz in 1963 to show why long-range weather forecasting is fundamentally limited. It is the founding example of chaos theory and is a standard test problem for numerical solvers, data assimilation and ML time-series forecasting.

$$\dot{x} = \sigma(y-x)$$
$$\dot{y} = x(\rho -z)-y$$
$$\dot{z} = xy-\beta z$$

State-space form of this can be given by:

$$\dot{\mathbf{x}} = f(\mathbf{x},t; \mathbf{b})$$

where

$$\mathbf{x} = \begin{bmatrix}
x \\ y \\ z
\end{bmatrix}; \;\;\;\; \mathbf{b} = \begin{bmatrix} \sigma \\ \rho \\ \beta \end{bmatrix}$$



In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.integrate import solve_ivp

# ---------------- INPUTS ----------------
b  = [10.0, 28.0, 8/3]     # [sigma, rho, beta]
x0 = [1.0, 1.0, 1.0]       # initial state [x, y, z]
dt = 0.01                  # output time step
T  = 50.0                  # total duration
method = "RK45"            # "RK45", "DOP853", "LSODA", "Radau", "BDF"
# ----------------------------------------

In [6]:
def lorenz(t, state, sigma, rho, beta):
    x, y, z = state
    return [sigma * (y - x),
            x * (rho - z) - y,
            x * y - beta * z]

t_eval = np.arange(0, T + dt / 2, dt)

sol = solve_ivp(lorenz, (0, T), x0, args=tuple(b),
                method=method, t_eval=t_eval,
                rtol=1e-9, atol=1e-12)
t = sol.t
x, y, z = sol.y

# --------- 3D Lorenz butterfly attractor visualisation ---------
fig3d = go.Figure(go.Scatter3d(
    x=x, y=y, z=z, mode="lines",
    line=dict(width=3, color=t, colorscale="Viridis",
              colorbar=dict(title="t"))
))
fig3d.update_layout(
    title=f"Lorenz attractor",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    height=650
)
fig3d.show()
# ----------------------------------------------------------------

#### Discrete-Time Systems

Dynamic equation for a continous autonomous system:

$$\dot{x} = f(x)$$

In a practical system, however, measurements are recorded at discrete time intervals and therefore, states are estimated at discrete time intervals.

$$x_{k+1} = F(x_k)$$

If $\Delta t$ is the time-step of an integrator, then the discrete-time dynamical system can be defined by:

$$x_k = x(k \cdot \Delta t)$$

Then, the flow map $F_{\Delta t}$ of states is given by:

$$x_{k+1} = x_k + \int_{k\Delta t}^{(k+1)\Delta t} f(x(\tau)) dt$$

In most dynamical systems of interest, computation of flow map $F_{\Delta t}$ solution in closed-form is not possible and must be adapted to certain numerical schemes such as Forward Euler, Runge-Kutta, etc.